# **PRAKTIKUM PERTEMUAN 10 DATA SCIENCE**

Nama : Ninis Indah Trisia

NIM : 250401020094

Kelas : IF405

### Muat dan Eksplorasi Data

In [18]:
import pandas as pd

# Membaca dataset
df = pd.read_csv("telco_churn.csv")

# Ukuran dataset
print("Ukuran Dataset:", df.shape)

# Lima data pertama
display(df.head())

# Informasi dataset
print("\nInformasi Dataset")
df.info()

# Missing value
print("\nMissing Value")
print(df.isnull().sum())

# Proporsi kelas churn
print("\nProporsi Kelas Churn")
print(df["Churn"].value_counts())

print("\nPersentase Kelas Churn")
print(df["Churn"].value_counts(normalize=True))

Ukuran Dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Informasi Dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 

### Preprocessing

In [19]:
from sklearn.model_selection import train_test_split

# 1. Encoding fitur kategorikal menjadi numerik [cite: 199, 201]
df_encoded = pd.get_dummies(df, drop_first=True)

# 2. Memisahkan Fitur (X) dan Target (y) [cite: 201]
if 'Churn_Yes' in df_encoded.columns:
    X = df_encoded.drop(columns=['Churn_Yes'])
    y = df_encoded['Churn_Yes']
else:
    X = df_encoded.drop(columns=['Churn'])
    y = df_encoded['Churn']

# 3. Membagi data menjadi data latih dan uji secara stratified [cite: 99, 101, 202, 203]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Data Latih: {X_tr.shape[0]} baris | Data Uji: {X_te.shape[0]} baris")

Data Latih: 5634 baris | Data Uji: 1409 baris


### Latih Model

In [20]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

print("Model berhasil dilatih.")

Model berhasil dilatih.


### Evaluasi

In [21]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi kelas
y_pred = rf.predict(X_te)

# Prediksi probabilitas
y_prob = rf.predict_proba(X_te)[:, 1]

# Classification Report
print(classification_report(
    y_te,
    y_pred,
    target_names=["Tidak Churn","Churn"]
))

# ROC-AUC
roc = roc_auc_score(y_te, y_prob)

print("ROC-AUC Score :", roc)

              precision    recall  f1-score   support

 Tidak Churn       0.84      0.90      0.87      1035
       Churn       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

ROC-AUC Score : 0.8337260068717869


### Prediksi Probabilitas dan Simpulkan

In [22]:
# Menghitung probabilitas churn
probabilitas = rf.predict_proba(X_te)[:, 1]

# Menampilkan hasil prediksi
hasil = pd.DataFrame({
    "Aktual": y_te.values,
    "Probabilitas Churn": probabilitas
})

hasil = hasil.sort_values(
    by="Probabilitas Churn",
    ascending=False
)

hasil.head(10)

,Aktual,Probabilitas Churn
1289,True,0.950000
618,True,0.933333
627,True,0.926667
34,True,0.913333
629,False,0.910000
788,True,0.910000
1252,True,0.910000
654,True,0.910000
995,False,0.903333
154,True,0.896667


# Kesimpulan
Berdasarkan hasil praktikum, dataset Telco Customer Churn terbukti mengalami masalah ketidakseimbangan kelas (imbalanced data) dengan proporsi pelanggan churn hanya sekitar 26,5%. Penggunaan parameter class_weight="balanced" pada algoritma Random Forest membantu model memberikan perhatian lebih kepada kelas minoritas tersebut saat proses pelatihan. Penerapan ini berhasil meningkatkan sensitivitas (recall) model dalam mendeteksi pelanggan yang berpotensi berhenti berlangganan, sehingga tim retensi dapat mengambil tindakan pencegahan secara dini. Secara keseluruhan, metrik ROC-AUC dan F1-score menunjukkan performa model yang stabil dan andal dalam membedakan karakteristik pelanggan yang bertahan maupun yang churn.